# 2 - Walk-forward & model training

**Stage 2 of 4.** Loads `model_df` from notebook 1 and does all the modelling:

1. Chronological train / validation / test split with a target-overlap embargo
2. Per-feature IC audit against the pre-stated sign hypotheses
3. The five-model ladder (OLS, Ridge, Lasso, ElasticNet, Random Forest) on the static split
4. **Embargoed multi-model walk-forward** - the out-of-sample engine; all five models refit per window
5. Traded signal = pre-registered walk-forward ElasticNet, plus rolling-IC and coefficient-stability charts

**Output** &rarr; `Data/interim/pred_df.parquet` (per-name OOS predictions + forward returns) and `Data/interim/best_params.json`, consumed by notebook **3 - portfolio sizing**.


In [ ]:
# --- Imports & setup -----------------------------------------
# Resolve paths from the repo root no matter where Jupyter launched.
import os
if os.path.basename(os.getcwd()) == "Notebooks":
    os.chdir("..")
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import spearmanr
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.6f}".format)


In [ ]:
# --- Load model_df + pipeline metadata from notebook 1 -------
model_df = pd.read_parquet("Data/interim/model_df.parquet")
model_df["date"] = pd.to_datetime(model_df["date"])

with open("Data/interim/pipeline_meta.json") as f:
    meta = json.load(f)

START_DATE           = meta["START_DATE"]
TARGET_HORIZON       = meta["TARGET_HORIZON"]
REBALANCE_EVERY      = meta["REBALANCE_EVERY"]
PERIODS_PER_YEAR     = meta["PERIODS_PER_YEAR"]
ALL_FEATURES         = meta["ALL_FEATURES"]
STOCK_FEATURES       = meta["STOCK_FEATURES"]
MACRO_ETF_FEATURES   = meta["MACRO_ETF_FEATURES"]
SHORT_VOL_FEATURES   = meta["SHORT_VOL_FEATURES"]
FUNDAMENTAL_FEATURES = meta["FUNDAMENTAL_FEATURES"]
SHORT_INT_FEATURES   = meta["SHORT_INT_FEATURES"]
FEATURE_HYPOTHESES   = {k: tuple(v) for k, v in meta["FEATURE_HYPOTHESES"].items()}

print(f"Loaded model_df {model_df.shape}  |  {len(ALL_FEATURES)} features")


In [ ]:
# =============================================================
# Cell 14 — Train / Validation / Test Split
# =============================================================
# Chronological split — NEVER random.
#
# SPLIT DATES:
# - Train:      START_DATE → 2022-12-31  (~3 years)
# - Validation: 2023-01-01 → 2023-12-31  (~1 year)
# - Test:       2024-01-01 → present     (~2+ years, held out)
#
# NOTE ON FEATURE AVAILABILITY:
# - Short interest EXCLUDED from ALL_FEATURES (see Cell 13) —
#   its Dec 2022+ only coverage would otherwise force-drop all
#   2020-2022 rows via the dropna below.
# - Short volume (Jan 2021+): available in most of train
# - Fundamentals (Jan 2021+): available in most of train
# Train set should now correctly span ~2021-2022 onwards
# (limited by short volume / fundamentals start date, not by
# short interest), instead of collapsing to December 2022 only.
#
# TARGET: next-day return (1-day forward)
# Justification: daily rebalancing strategy; 1-day return
# is the most direct and unambiguous prediction target.
# =============================================================


TRAIN_END = "2022-12-31"
VAL_END   = "2023-12-31"

train_df = model_df[model_df["date"] <= TRAIN_END].copy()
val_df   = model_df[(model_df["date"] > TRAIN_END) & (model_df["date"] <= VAL_END)].copy()
test_df  = model_df[model_df["date"] > VAL_END].copy()

# ── v19 PURGE / EMBARGO (lookahead fix) ──
# With an H-day forward target, rows on the last H dates of train have
# targets computed from returns INSIDE the validation window (and the
# last H dates of validation reach into test). That is genuine leakage
# across split boundaries. Drop the last TARGET_HORIZON dates of train
# and of validation.
_tr_dates = np.sort(train_df["date"].unique())
train_df  = train_df[train_df["date"] <= _tr_dates[-(TARGET_HORIZON + 1)]].copy()
_va_dates = np.sort(val_df["date"].unique())
val_df    = val_df[val_df["date"] <= _va_dates[-(TARGET_HORIZON + 1)]].copy()
print(f"Cell 14: Purged last {TARGET_HORIZON} dates of train and of validation"
      f" (target-overlap embargo).")

print("="*60)
print("Cell 14: TRAIN / VALIDATION / TEST SPLIT")
print("="*60)
print(f"Cell 14: Train:      {train_df['date'].min().date()} → {train_df['date'].max().date()}"
      f"  |  {len(train_df):,} rows  |  {train_df['ticker'].nunique()} tickers")
print(f"Cell 14: Validation: {val_df['date'].min().date()} → {val_df['date'].max().date()}"
      f"  |  {len(val_df):,} rows  |  {val_df['ticker'].nunique()} tickers")
print(f"Cell 14: Test:       {test_df['date'].min().date()} → {test_df['date'].max().date()}"
      f"  |  {len(test_df):,} rows  |  {test_df['ticker'].nunique()} tickers")
print(f"\nCell 13: Target stats (train set):")
print(train_df["target"].describe().to_string())
print(f"\nCell 13: Feature availability in train set (non-null %):")
for grp, cols in [("Stock",STOCK_FEATURES),("Macro/ETF",MACRO_ETF_FEATURES),
                   ("Short Vol",SHORT_VOL_FEATURES),
                   ("Fundamentals",FUNDAMENTAL_FEATURES)]:
    avail = [c for c in cols if c in train_df.columns]
    if avail:
        pct = train_df[avail].notna().mean().mean() * 100
        print(f"  {grp:<15}: {pct:.1f}% available")

In [ ]:
# =============================================================
# Cell 15 — IC / ICIR Helper Functions
# =============================================================
# Standard quant evaluation metrics used throughout.
#
# IC (Information Coefficient):
#   Spearman rank correlation between predicted and actual
#   next-day returns, computed cross-sectionally per day.
#   WHY SPEARMAN: robust to outliers, measures ranking not scale.
#
# IC interpretation:
#   |IC| > 0.05 → strong (rare)
#   |IC| > 0.02 → usable in ensemble
#   |IC| ~ 0    → no predictive power
#
# ICIR = IC Mean / IC Std (consistency of signal)
# Hit Rate = % of days with positive IC (should be > 50%)
# =============================================================

def compute_daily_ic(df, pred_col="pred", target_col="target", min_stocks=30):
    """
    Compute per-day Spearman IC between predictions and targets.
    Returns pd.Series indexed by date.
    """
    records = []
    for date, g in df.groupby("date"):
        g = g[[pred_col, target_col]].dropna()
        if len(g) < min_stocks or g[pred_col].nunique() < 2:
            continue
        ic, _ = spearmanr(g[pred_col], g[target_col])
        if np.isfinite(ic):
            records.append({"date": date, "ic": ic})
    if not records:
        return pd.Series(dtype=float)
    return pd.DataFrame(records).set_index("date")["ic"]

def ic_summary(ic_series, label=""):
    """Print IC / ICIR / hit rate summary with cell label."""
    if len(ic_series) == 0:
        print(f"  {label}: No valid IC observations.")
        return {}
    mean_ic  = ic_series.mean()
    std_ic   = ic_series.std()
    icir     = mean_ic / (std_ic + 1e-9)
    hit_rate = (ic_series > 0).mean()
    # v18: too few valid IC days ⇒ statistically meaningless summary.
    # (v17's RF validation "IC=+0.12" was computed on a single day.)
    flag = "" if len(ic_series) >= 30 else "  ← INVALID (<30 valid days)"
    print(f"  {label:<25}  IC={mean_ic:+.5f}  Std={std_ic:.5f}  "
          f"ICIR={icir:+.3f}  Hit={hit_rate:.1%}  N={len(ic_series)}{flag}")
    return {"ic_mean":mean_ic,"ic_std":std_ic,"icir":icir,"hit_rate":hit_rate}

def sharpe(daily_returns, ann_factor=252):
    """Annualised Sharpe ratio from a daily return series."""
    mu, sig = daily_returns.mean(), daily_returns.std()
    return 0.0 if sig < 1e-10 else mu / sig * np.sqrt(ann_factor)

print("Cell 15: IC/ICIR/Sharpe helper functions defined.")

In [ ]:
# =============================================================
# Cell 16 — Feature-Level IC Audit (Signal Discovery)
# =============================================================
# Before fitting any model, we audit each individual feature
# for raw predictive power on the TRAINING SET ONLY.
#
# This is good research hygiene:
# - Confirms which features carry genuine cross-sectional signal
# - Identifies noise features (IC ~ 0)
# - Guides model selection and feature grouping
#
# NOTE on macro/ETF features: these are identical for all stocks
# on a given day, so Spearman IC may show "insufficient data"
# or very small values. This does not mean they are useless —
# they interact with stock features in the model.
# =============================================================

print("="*60)
print("Cell 16: FEATURE-LEVEL IC AUDIT (Training Set)")
print("="*60)
print(f"  {'Feature':<42}  IC      Std     ICIR    Hit%  Days  Hyp  Match")

ic_by_feature = {}

for feat in ALL_FEATURES:
    if feat not in train_df.columns:
        continue
    tmp = train_df[["date",feat,"target"]].rename(columns={feat:"pred"})
    ic_s = compute_daily_ic(tmp, min_stocks=50)
    if len(ic_s) == 0:
        print(f"  {'⚠ '+feat:<42}  insufficient data")
        continue
    mean_ic = ic_s.mean()
    std_ic  = ic_s.std()
    icir    = mean_ic / (std_ic + 1e-9)
    hit     = (ic_s > 0).mean()
    # v18: compare realised IC sign against the pre-stated hypothesis
    hyp = FEATURE_HYPOTHESES.get(feat, (None, ""))[0]
    if hyp is None:
        match = " ? "
    else:
        realised = "+" if mean_ic > 0 else "-"
        match = " ✓ " if realised == hyp else " ✗ "
    print(f"  {feat:<42}  {mean_ic:+.4f}  {std_ic:.4f}  "
          f"{icir:+.3f}  {hit:.1%}  {len(ic_s):>4}   {hyp or '?'}   {match}")
    ic_by_feature[feat] = {"ic_mean":mean_ic,"icir":icir,"hyp":hyp}

_checked = [v for v in ic_by_feature.values() if v.get("hyp")]
_agree   = sum(1 for v in _checked
               if ("+" if v["ic_mean"] > 0 else "-") == v["hyp"])
print(f"\nCell 16: Hypothesis sign agreement: {_agree}/{len(_checked)} features")
print("Cell 16: (Disagreements are findings, not failures — report them.)")
print("Cell 16: IC values near zero are expected for individual features;")
print("Cell 16: the model aggregates weak signals across features.")

In [ ]:
# =============================================================
# Cell 17 — Model Level 1: OLS (Ordinary Least Squares)
# =============================================================
# Baseline linear model. Performance floor for all other models.
#
# WHY START WITH OLS:
# - Fully interpretable coefficients (sign + magnitude)
# - Stable, no hyperparameters
# - If OLS IC ~ 0, more complex models are unlikely to help
#
# V12 vs V10: OLS now has access to 19 features (vs 12 in v10),
# residualized against the market factor (Cell 12).
# Coefficient signs tell us about the economic relationship:
# - Momentum should be +ve (trend following)
# - Vol should be -ve (high vol = mean reversion)
# - Short ratio z-score should be -ve (high shorting = bearish)
# - ROE should be +ve (quality premium)
# - Revenue growth should be +ve (growth premium)
# =============================================================

print("="*60)
print("Cell 17: MODEL 1 — OLS (Ordinary Least Squares)")
print("="*60)

X_train = train_df[ALL_FEATURES].values
y_train = train_df["target"].values
X_val   = val_df[ALL_FEATURES].values

ols_model = LinearRegression()
print("Cell 17: Fitting OLS on training set...")
ols_model.fit(X_train, y_train)

train_df = train_df.copy()
val_df   = val_df.copy()
train_df["pred_ols"] = ols_model.predict(X_train)
val_df["pred_ols"]   = ols_model.predict(X_val)

print("\nCell 16: --- IC on Training Set ---")
ic_train_ols = compute_daily_ic(train_df, pred_col="pred_ols")
ic_summary(ic_train_ols, "OLS Train")

print("\nCell 16: --- IC on Validation Set ---")
ic_val_ols = compute_daily_ic(val_df, pred_col="pred_ols")
ic_summary(ic_val_ols, "OLS Val")

print("\nCell 16: --- OLS Coefficients (sorted by magnitude) ---")
for feat, coef in sorted(zip(ALL_FEATURES, ols_model.coef_), key=lambda x: abs(x[1]), reverse=True):
    direction = "↑" if coef > 0 else "↓"
    print(f"  {direction}  {feat:<42}  {coef:+.6f}")

In [ ]:
# =============================================================
# Cell 18 — Model Level 2: Ridge Regression
# =============================================================
# Adds L2 penalty: minimise RSS + λ * ||β||²
#
# WHY RIDGE:
# With 19 features including correlated momentum signals
# (mom_5, mom_20, mom_60, mom_252) and correlated short signals,
# OLS can assign unstable large opposite-sign coefficients.
# Ridge shrinks all coefficients toward zero, stabilising them.
#
# Ridge KEEPS all features but reduces their magnitude.
# Especially useful here because all feature groups (momentum,
# short vol, fundamentals) likely each carry some signal.
# =============================================================

print("="*60)
print("Cell 18: MODEL 2 — Ridge Regression")
print("="*60)

best_ridge_ic = -np.inf
best_ridge_alpha = None
best_ridge_model = None

for alpha in [0.01, 0.1, 1.0, 10.0]:
    m = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    m.fit(train_df[ALL_FEATURES], train_df["target"])
    preds = m.predict(val_df[ALL_FEATURES])
    tmp = val_df.copy()
    tmp["pred_tmp"] = preds
    ic_s = compute_daily_ic(tmp, pred_col="pred_tmp")
    mean_ic = ic_s.mean() if len(ic_s) > 0 else -999
    print(f"  Cell 18: alpha={alpha:<6}  Val IC={mean_ic:+.5f}")
    if mean_ic > best_ridge_ic:
        best_ridge_ic    = mean_ic
        best_ridge_alpha = alpha
        best_ridge_model = m

print(f"\nCell 17: Best Ridge alpha={best_ridge_alpha}  (Val IC={best_ridge_ic:+.5f})")
train_df["pred_ridge"] = best_ridge_model.predict(train_df[ALL_FEATURES])
val_df["pred_ridge"]   = best_ridge_model.predict(val_df[ALL_FEATURES])

print("\nCell 17: --- IC Summary (best Ridge) ---")
ic_val_ridge = compute_daily_ic(val_df, pred_col="pred_ridge")
ic_summary(ic_val_ridge, "Ridge Val")

In [ ]:
# =============================================================
# Cell 19 — Model Level 3: Lasso Regression
# =============================================================
# Adds L1 penalty: minimise RSS + λ * ||β||₁
# The L1 penalty drives some coefficients exactly to zero —
# automatic feature selection.
#
# WHY LASSO:
# With 19 features across 4 different data sources (short interest excluded), it is
# plausible that not all features add value. Lasso tells us
# which features the data itself considers informative.
#
# IMPORTANT FINDING TO WATCH FOR:
# If Lasso zeroes out entire feature groups (e.g. all
# fundamental features), that suggests those signals don't
# add cross-sectional predictive power beyond what the
# momentum/short signals already capture.
# =============================================================

print("="*60)
print("Cell 19: MODEL 3 — Lasso Regression")
print("="*60)

best_lasso_ic    = -np.inf
best_lasso_alpha = None
best_lasso_model = None

for alpha in [1e-5, 1e-4, 1e-3, 0.01, 0.1]:
    m = make_pipeline(StandardScaler(), Lasso(alpha=alpha, max_iter=5000))
    m.fit(train_df[ALL_FEATURES], train_df["target"])
    preds = m.predict(val_df[ALL_FEATURES])
    tmp = val_df.copy()
    tmp["pred_tmp"] = preds
    ic_s = compute_daily_ic(tmp, pred_col="pred_tmp")
    mean_ic = ic_s.mean() if len(ic_s) > 0 else -999
    coef = m.named_steps["lasso"].coef_
    n_nz = np.sum(coef != 0)
    print(f"  Cell 19: alpha={alpha:<8}  Val IC={mean_ic:+.5f}  Non-zero: {n_nz}/{len(ALL_FEATURES)}")
    if mean_ic > best_lasso_ic:
        best_lasso_ic    = mean_ic
        best_lasso_alpha = alpha
        best_lasso_model = m

print(f"\nCell 18: Best Lasso alpha={best_lasso_alpha}  (Val IC={best_lasso_ic:+.5f})")
train_df["pred_lasso"] = best_lasso_model.predict(train_df[ALL_FEATURES])
val_df["pred_lasso"]   = best_lasso_model.predict(val_df[ALL_FEATURES])

lasso_coef = best_lasso_model.named_steps["lasso"].coef_
print("\nCell 18: --- Features selected by Lasso (non-zero) ---")
for feat, coef in zip(ALL_FEATURES, lasso_coef):
    if coef != 0:
        print(f"  {'↑' if coef>0 else '↓'}  {feat:<42}  {coef:+.6f}")

print("\nCell 18: --- IC Summary (best Lasso) ---")
ic_val_lasso = compute_daily_ic(val_df, pred_col="pred_lasso")
ic_summary(ic_val_lasso, "Lasso Val")

In [ ]:
# =============================================================
# Cell 20 — Model Level 4: ElasticNet
# =============================================================
# Combines L1 + L2: minimise RSS + λ[l1_ratio*||β||₁ + (1-l1_ratio)*||β||²]
#
# WHY ELASTICNET:
# "If you don't know whether to use Ridge or Lasso, use ElasticNet."
# — course material. With 19 features across multiple correlated
# groups, ElasticNet handles both the collinearity issue (L2)
# and the feature selection issue (L1) simultaneously.
#
# This is our primary walk-forward model.
# Grid search over alpha and l1_ratio to find best combination.
# =============================================================

print("="*60)
print("Cell 20: MODEL 4 — ElasticNet")
print("="*60)

best_en_ic     = -np.inf
best_en_params = None
best_en_model  = None

for alpha in [1e-4, 1e-3, 0.01]:
    for l1r in [0.3, 0.5, 0.7]:
        m = make_pipeline(
            StandardScaler(),
            ElasticNet(alpha=alpha, l1_ratio=l1r, max_iter=5000)
        )
        m.fit(train_df[ALL_FEATURES], train_df["target"])
        preds = m.predict(val_df[ALL_FEATURES])
        tmp = val_df.copy()
        tmp["pred_tmp"] = preds
        ic_s = compute_daily_ic(tmp, pred_col="pred_tmp")
        mean_ic = ic_s.mean() if len(ic_s) > 0 else -999
        print(f"  Cell 20: alpha={alpha:<7}  l1_ratio={l1r}  Val IC={mean_ic:+.5f}")
        if mean_ic > best_en_ic:
            best_en_ic     = mean_ic
            best_en_params = (alpha, l1r)
            best_en_model  = m

print(f"\nCell 19: Best ElasticNet: alpha={best_en_params[0]}, l1_ratio={best_en_params[1]}"
      f"  (Val IC={best_en_ic:+.5f})")
train_df["pred_en"] = best_en_model.predict(train_df[ALL_FEATURES])
val_df["pred_en"]   = best_en_model.predict(val_df[ALL_FEATURES])

print("\nCell 19: --- IC Summary (best ElasticNet) ---")
ic_val_en = compute_daily_ic(val_df, pred_col="pred_en")
ic_summary(ic_val_en, "ElasticNet Val")

In [ ]:
# =============================================================
# Cell 21 — Model Level 5: Random Forest
# =============================================================
# Level 4 supervised ML model. Captures non-linear interactions.
#
# WHY RF IN V11:
# With 19 features from 4 data sources, interaction effects
# become more plausible:
# - High short ratio + low momentum = stronger bearish signal
# - High ROE + high revenue growth = stronger quality signal
# - Fundamental signals may only matter in certain macro regimes
# RF can learn these interactions; linear models cannot.
#
# OVERFITTING CONTROLS (deliberately conservative):
# - max_depth=4:          shallow trees
# - min_samples_leaf=500: large minimum leaf size
# - n_estimators=100:     sufficient but not excessive
#
# Feature importances tell us which data sources the RF
# considers most informative — a key analytical finding.
# =============================================================

print("="*60)
print("Cell 21: MODEL 5 — Random Forest")
print("="*60)
print("Cell 21: Training Random Forest (1-2 minutes)...")

rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=4,
    min_samples_leaf=500,
    n_jobs=-1,
    random_state=42
)
rf_model.fit(train_df[ALL_FEATURES], train_df["target"])
print("Cell 21: Training complete.")

train_df["pred_rf"] = rf_model.predict(train_df[ALL_FEATURES])
val_df["pred_rf"]   = rf_model.predict(val_df[ALL_FEATURES])

print("\nCell 20: --- IC Summary (Random Forest) ---")
ic_val_rf = compute_daily_ic(val_df, pred_col="pred_rf")
ic_summary(ic_val_rf, "RandomForest Val")

print("\nCell 20: --- Feature Importances by Data Source ---")
importances = pd.Series(rf_model.feature_importances_, index=ALL_FEATURES)
# Group by data source
groups = [
    ("Stock (momentum/vol)", STOCK_FEATURES),
    ("Macro/ETF",            MACRO_ETF_FEATURES),
    ("Short Volume",         SHORT_VOL_FEATURES),
    ("Fundamentals",         FUNDAMENTAL_FEATURES),
]
for grp_name, grp_cols in groups:
    avail = [c for c in grp_cols if c in ALL_FEATURES]
    grp_imp = importances[avail].sum()
    print(f"  {grp_name:<30}  Total importance: {grp_imp:.4f}")

print("\nCell 20: --- Top 10 individual features ---")
for feat, imp in importances.sort_values(ascending=False).head(10).items():
    bar = "█" * int(imp * 200)
    print(f"  {feat:<42}  {imp:.4f}  {bar}")

# Plotted version of feature importances (in addition to text above)
import matplotlib.pyplot as plt

print("\nCell 21: Plotting feature importances...")
top_importances = importances.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_importances.index[::-1], top_importances.values[::-1], color="steelblue")
ax.set_title("Random Forest — Top 15 Feature Importances")
ax.set_xlabel("Importance")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig("Results/feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print("Cell 21: Chart saved to Results/feature_importance.png")

In [ ]:
# =============================================================
# Cell 22 — Model Comparison Summary
# =============================================================
# Side-by-side comparison of all five models on validation set.
# Higher ICIR (not just IC mean) is preferred — consistency
# matters more than a few exceptional days.
#
# V11 vs V10 comparison:
# We expect IC values to improve due to new features (short vol,
# fundamentals). If they don't, the new features may not add
# cross-sectional predictive power on this dataset/period.
# Either finding is valid and worth reporting.
# =============================================================

print("="*60)
print("Cell 22: MODEL COMPARISON SUMMARY (Validation Set)")
print("="*60)
print(f"  {'Model':<20}  IC Mean   IC Std   ICIR     Hit Rate")
print("  " + "-"*55)

model_results = {}
for label, pred_col in [
    ("OLS",          "pred_ols"),
    ("Ridge",        "pred_ridge"),
    ("Lasso",        "pred_lasso"),
    ("ElasticNet",   "pred_en"),
    ("RandomForest", "pred_rf"),
]:
    ic_s = compute_daily_ic(val_df, pred_col=pred_col)
    if len(ic_s) == 0:
        print(f"  {label:<20}  (no valid IC)")
        continue
    mean_ic = ic_s.mean()
    std_ic  = ic_s.std()
    icir    = mean_ic / (std_ic + 1e-9)
    hit     = (ic_s > 0).mean()
    # v18: flag statistically invalid rows (v17's RF row was N=1)
    flag = "" if len(ic_s) >= 30 else f"  ← INVALID (N={len(ic_s)} days)"
    print(f"  {label:<20}  {mean_ic:+.5f}   {std_ic:.5f}  {icir:+.4f}  {hit:.1%}{flag}")
    model_results[label] = {"ic_mean":mean_ic,"icir":icir,"hit":hit,"ic_series":ic_s}

print("\nCell 21: IC values of 0.01-0.03 are typical for individual models.")
print("Cell 22: The forecast combination in Cell 23 aims to improve ICIR stability.")

In [ ]:
# =============================================================
# Cell 23 — Signal Selection (PRE-REGISTERED — replaces v17's
#           IC-weighted static combination)
# =============================================================
# v17 combined the five models with weights fitted to validation
# ICIR, and v16/v17 additionally re-fitted those weights through
# time ("dynamic combination"). Both are hypothesis-free: they
# select whatever worked recently, which is a recipe for fitting
# noise and an indefensible answer to "why these weights?".
#
# v18 REPLACES fitted combination weights with a pre-registered
# choice, stated before looking at out-of-sample results:
#
#   TRADED SIGNAL: walk-forward ElasticNet (Cell 24 → Cell 25).
#   WHY, A PRIORI:
#   - the feature set has correlated blocks (4 momentum horizons,
#     3 volatility measures): L2 stabilises within blocks, L1
#     prunes across them — exactly ElasticNet's design case;
#   - a linear model's coefficients can be audited for sign and
#     stability across retraining windows (Cell 27), so the
#     hypothesis test extends to the model's internals;
#   - Random Forest is retained as a NON-TRADED diagnostic for
#     non-linearity (if RF's OOS IC ≫ linear OOS IC, the linear
#     hypothesis is incomplete — that is the only claim it tests).
#
#   ROBUSTNESS CHECK (not a traded alternative): the 1/N average
#   of the four linear models. 1/N has NO fitted weights, so it
#   cannot chase recent performance; if the pre-registered
#   ElasticNet and 1/N disagree wildly out-of-sample, the choice
#   of linear model mattered more than the signal — worth knowing.
#
# This cell reports the validation-set ICs for the record. No
# weights are fitted, and nothing here feeds position sizing.
# =============================================================

print("=" * 60)
print("Cell 23: SIGNAL SELECTION (pre-registered — no fitted weights)")
print("=" * 60)

LINEAR_PRED_COLS = ["pred_ols", "pred_ridge", "pred_lasso", "pred_en"]

for df_split in [train_df, val_df]:
    df_split["pred_1n"] = df_split[LINEAR_PRED_COLS].mean(axis=1)

print("\nCell 23: --- Validation-set IC, for the record ---")
ic_summary(compute_daily_ic(val_df, pred_col="pred_en"),  "ElasticNet (traded)")
ic_summary(compute_daily_ic(val_df, pred_col="pred_1n"),  "1/N linear (robustness)")
ic_summary(compute_daily_ic(val_df, pred_col="pred_rf"),  "RandomForest (diagnostic)")

print("\nCell 23: Traded signal is fixed a priori: walk-forward ElasticNet.")
print("Cell 23: No IC-weighted or dynamic combination exists in v18.")

In [ ]:
# =============================================================
# Cell 24 — Multi-Model Walk-Forward Backtest
# =============================================================
# Proper out-of-sample evaluation via walk-forward retraining,
# for ALL FIVE models — not just ElasticNet.
#
# WHY THIS CHANGED FROM v12:
# v12 only walk-forward-tested ElasticNet, on the reasoning that
# it was "best balance of stability and signal." But that choice
# was itself made by looking at static validation-period IC
# (Cell 22's Model Comparison) — meaning the validation period
# was implicitly used twice: once to pick ElasticNet, and again
# inside the walk-forward that only tests ElasticNet. v15
# (a colleague's parallel version) identified this exact issue
# and proposed evaluating every model out-of-sample, then
# dynamically combining them — a more defensible research
# design, since model selection happens out-of-sample too.
#
# LOGIC:
# - Training window: 2 years (504 trading days)
# - Step: 21 trading days (~1 month)
# - At each step: fit ALL FIVE models on the past 2 years,
#   predict the next 21 days for each, roll forward
# - ElasticNet's coefficients are additionally tracked per
#   window (coef_history) for the stability chart in Cell 26 —
#   coefficients are only meaningful for linear models, so this
#   stays specific to ElasticNet even though all 5 models run.
#
# WHY ALL FIVE MODELS COSTS LITTLE EXTRA:
# OLS/Ridge/Lasso/ElasticNet are near-instant to fit even at
# this data volume. Random Forest is the slow one, but at
# n_estimators=50 (reduced from 100 used in the static ladder)
# the full loop still completes in a reasonable time.
#
# FEATURE AVAILABILITY NOTE:
# Early windows (2020-2021) won't have short interest data —
# moot here since short interest is excluded from ALL_FEATURES
# entirely (see Cell 13). Short volume / fundamentals (2021+)
# do constrain how far back usable windows start.
#
# OUTPUT: pred_df — canonical prediction dataset for the
# downstream dynamic combination (Cell 25) and portfolio
# construction. Contains one prediction column per model:
# pred_ols, pred_ridge, pred_lasso, pred_en, pred_rf.
# =============================================================

print("="*60)
print("Cell 24: MULTI-MODEL WALK-FORWARD BACKTEST")
print("="*60)

TRAIN_WINDOW = 504
STEP         = 21

df_wf = model_df.copy()
df_wf = df_wf.sort_values(["date","ticker"]).reset_index(drop=True)
unique_dates = np.sort(df_wf["date"].unique())

all_predictions = []
n_windows = 0
n_failed  = 0
coef_history = []   # tracks ElasticNet coefficients per window for stability analysis

print(f"Cell 24: Total trading days: {len(unique_dates)}")
print(f"Cell 24: Training window: {TRAIN_WINDOW} days | Step: {STEP} days")
print(f"Cell 24: Expected windows: ~{(len(unique_dates)-TRAIN_WINDOW)//STEP}")
print("Cell 24: Fitting 5 models per window: OLS, Ridge, Lasso, ElasticNet, RandomForest")
print("Cell 24: Running walk-forward loop...")

for i in range(TRAIN_WINDOW, len(unique_dates), STEP):
    n_windows += 1
    # v19 PURGE (lookahead fix): the H-day targets of the last H
    # training dates use returns INSIDE this window's test period —
    # exclude those dates from training (embargo).
    train_dates = unique_dates[i - TRAIN_WINDOW : i - TARGET_HORIZON]
    test_dates  = unique_dates[i : i + STEP]

    tr = df_wf[df_wf["date"].isin(train_dates)]
    te = df_wf[df_wf["date"].isin(test_dates)]

    if len(tr) < 5000 or len(te) < 50:
        n_failed += 1
        continue

    tr_clean = tr.dropna(subset=ALL_FEATURES + ["target"])
    te_clean = te.dropna(subset=ALL_FEATURES)

    if len(tr_clean) < 2000:
        n_failed += 1
        continue

    try:
        Xtr = tr_clean[ALL_FEATURES]
        ytr = tr_clean["target"]
        Xte = te_clean[ALL_FEATURES]

        te_out = te_clean.copy()

        # --- OLS ---
        m_ols = LinearRegression()
        m_ols.fit(Xtr, ytr)
        te_out["pred_ols"] = m_ols.predict(Xte)

        # --- Ridge (fixed alpha from static ladder, Cell 18) ---
        m_ridge = make_pipeline(StandardScaler(), Ridge(alpha=best_ridge_alpha))
        m_ridge.fit(Xtr, ytr)
        te_out["pred_ridge"] = m_ridge.predict(Xte)

        # --- Lasso (fixed alpha from static ladder, Cell 19) ---
        m_lasso = make_pipeline(StandardScaler(), Lasso(alpha=best_lasso_alpha, max_iter=5000))
        m_lasso.fit(Xtr, ytr)
        te_out["pred_lasso"] = m_lasso.predict(Xte)

        # --- ElasticNet (fixed params from static ladder, Cell 20) ---
        m_en = make_pipeline(
            StandardScaler(),
            ElasticNet(alpha=best_en_params[0], l1_ratio=best_en_params[1], max_iter=5000)
        )
        m_en.fit(Xtr, ytr)
        te_out["pred_en"] = m_en.predict(Xte)

        # --- Random Forest (reduced n_estimators for walk-forward speed) ---
        m_rf = RandomForestRegressor(
            n_estimators=50, max_depth=4, min_samples_leaf=500,
            n_jobs=-1, random_state=42
        )
        m_rf.fit(Xtr, ytr)
        te_out["pred_rf"] = m_rf.predict(Xte)

        all_predictions.append(te_out)

        # Capture ElasticNet coefficients for stability analysis (Cell 26)
        coef_history.append({
            "train_end": train_dates[-1],
            **dict(zip(ALL_FEATURES, m_en.named_steps["elasticnet"].coef_))
        })

        if n_windows % 5 == 0:
            print(f"  Cell 24: Window {n_windows}: train end "
                  f"{train_dates[-1].astype('datetime64[D]')} | test: {len(te_out)} rows")
    except Exception as e:
        print(f"  Cell 24: Window {n_windows} failed: {e}")
        n_failed += 1

pred_df = pd.concat(all_predictions, ignore_index=True)
pred_df = pred_df.sort_values(["date","ticker"]).reset_index(drop=True)

print(f"\nCell 24: Walk-forward complete.")
print(f"Cell 24: Windows run: {n_windows} | Failed: {n_failed}")
print(f"Cell 24: Prediction dataset: {pred_df.shape}")
print(f"Cell 24: Date range: {pred_df['date'].min().date()} → {pred_df['date'].max().date()}")

print("\nCell 24: --- Walk-Forward Out-of-Sample IC, per model ---")
wf_model_results = {}
for label, pred_col in [
    ("OLS",          "pred_ols"),
    ("Ridge",        "pred_ridge"),
    ("Lasso",        "pred_lasso"),
    ("ElasticNet",   "pred_en"),
    ("RandomForest", "pred_rf"),
]:
    ic_s = compute_daily_ic(pred_df, pred_col=pred_col)
    res = ic_summary(ic_s, f"WF {label} OOS")
    if res:
        wf_model_results[label] = {**res, "ic_series": ic_s}

In [ ]:
# =============================================================
# Cell 25 — Traded Signal Definition (replaces v17's dynamic
#           forecast combination)
# =============================================================
# v17's trailing-126-day ICIR re-weighting is REMOVED (see Cell
# 23 for the rationale). The traded signal is simply the
# pre-registered walk-forward ElasticNet prediction. The column
# is still named `pred_ensemble` so every downstream cell
# (rolling IC chart, portfolio construction, costs, summary)
# runs unchanged.
#
# Alongside it, two non-traded reference signals are evaluated
# on the SAME out-of-sample walk-forward window:
#   - 1/N average of the four linear models (no fitted weights)
#   - Random Forest (non-linearity diagnostic)
# If the pre-registered signal and 1/N diverge materially, model
# choice dominates the result; if RF materially beats the linear
# models, the linearity assumption is the binding constraint.
# Either finding is reported as-is.
# =============================================================

print("=" * 60)
print("Cell 25: TRADED SIGNAL — pre-registered walk-forward ElasticNet")
print("=" * 60)

pred_df = pred_df.sort_values(["date", "sec_id"]).reset_index(drop=True)

# Traded signal (pre-registered)
pred_df["pred_ensemble"] = pred_df["pred_en"]

# Non-traded references on the same OOS window
pred_df["pred_1n"] = pred_df[["pred_ols", "pred_ridge",
                              "pred_lasso", "pred_en"]].mean(axis=1)

print("\nCell 25: --- Out-of-sample IC (walk-forward window) ---")
ic_traded = compute_daily_ic(pred_df, pred_col="pred_ensemble")
ic_summary(ic_traded, "ElasticNet (TRADED)")
ic_summary(compute_daily_ic(pred_df, pred_col="pred_1n"), "1/N linear (reference)")
ic_summary(compute_daily_ic(pred_df, pred_col="pred_rf"), "RandomForest (diagnostic)")

if len(ic_traded) > 0:
    _tstat = ic_traded.mean() / (ic_traded.std() / np.sqrt(len(ic_traded)) + 1e-12)
    print(f"\nCell 25: Traded-signal IC t-stat: {_tstat:+.2f}"
          f"  (|t| < ~2 ⇒ IC not statistically distinguishable from zero;")
    print("Cell 25:  that finding, if it occurs, is reported as the result.)")

In [ ]:
# =============================================================
# Cell 26 — Rolling IC Chart
# =============================================================
# Visualizes how the TRADED SIGNAL's (walk-forward ElasticNet)
# (IC) evolves over time. A healthy signal should hover
# consistently above zero. Sharp drops or sign flips indicate
# periods where the model's signal broke down -- useful for
# spotting regime changes or data quality issues.
# =============================================================

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

print("Cell 26: Plotting rolling IC over the walk-forward period...")

ic_wf_series = compute_daily_ic(pred_df, pred_col="pred_ensemble")
rolling_ic_20d = ic_wf_series.rolling(20, min_periods=10).mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(ic_wf_series.index, ic_wf_series.values,
        color="lightgray", linewidth=0.8, label="Daily IC")
ax.plot(rolling_ic_20d.index, rolling_ic_20d.values,
        color="steelblue", linewidth=2, label="20-day Rolling IC")
ax.axhline(0, color="black", linestyle="--", linewidth=0.8)
ax.set_title("Traded Signal (Walk-Forward ElasticNet): Rolling Information Coefficient")
ax.set_ylabel("Spearman IC")
ax.legend(loc="upper left", fontsize=9)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.savefig("Results/rolling_ic.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Cell 26: Chart saved to Results/rolling_ic.png")
print(f"Cell 26: Percent of days with positive 20-day rolling IC: {(rolling_ic_20d > 0).mean():.1%}")

In [ ]:
# =============================================================
# Cell 27 — Coefficient Stability Chart
# =============================================================
# The assignment explicitly states:
# "Coefficients produced should be relatively consistent
#  across time" (course material, Level 3 models section).
#
# This chart shows how each ElasticNet coefficient evolved
# across all walk-forward retraining windows. Stable,
# consistently-signed coefficients indicate a genuine, durable
# relationship. Coefficients that flip sign or swing wildly
# between windows suggest overfitting to short-term noise
# rather than a real structural signal.
# =============================================================

print("Cell 27: Plotting coefficient stability across walk-forward windows...")

coef_df = pd.DataFrame(coef_history).set_index("train_end")

# Plot the 6 features with the largest average absolute coefficient
top_features = coef_df.abs().mean().sort_values(ascending=False).head(6).index

fig, ax = plt.subplots(figsize=(14, 6))
for feat in top_features:
    ax.plot(coef_df.index, coef_df[feat], marker="o", markersize=3,
            linewidth=1.2, label=feat)
ax.axhline(0, color="black", linestyle="--", linewidth=0.8)
ax.set_title("ElasticNet Coefficient Stability Across Walk-Forward Windows\n"
             "(Top 6 features by average magnitude)")
ax.set_ylabel("Coefficient Value")
ax.legend(loc="best", fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.savefig("Results/coefficient_stability.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Cell 27: Chart saved to Results/coefficient_stability.png")
print("\nCell 27: Sign consistency check (% of windows where sign matches the most common sign):")
for feat in ALL_FEATURES:
    if feat in coef_df.columns:
        signs = np.sign(coef_df[feat])
        most_common_sign = signs.mode()[0] if len(signs.mode()) > 0 else 0
        consistency = (signs == most_common_sign).mean()
        print(f"  {feat:<25}  {consistency:.1%} consistent")

In [ ]:
# =============================================================
# Persist walk-forward predictions + tuned params for notebook 3
# =============================================================
keep = ["date", "ticker", "sec_id",
        "pred_ols", "pred_ridge", "pred_lasso", "pred_en", "pred_rf",
        "pred_ensemble", "pred_1n",
        "target", "ret", "fwd_ret_1d", "fwd_ret_h", "tc_sigma", "tc_adv"]
keep = [c for c in keep if c in pred_df.columns]
pred_df[keep].to_parquet("Data/interim/pred_df.parquet", index=False)

best_params = {
    "best_ridge_alpha": best_ridge_alpha,
    "best_lasso_alpha": best_lasso_alpha,
    "best_en_params":   list(best_en_params),
}
with open("Data/interim/best_params.json", "w") as f:
    json.dump(best_params, f, indent=2)

print("Saved -> Data/interim/pred_df.parquet", pred_df[keep].shape)
print("Saved -> Data/interim/best_params.json", best_params)
